# AML - LEAVE ONE VARIANT OUT 
This script submits the commands for the leave-one-variant-out (LOVO) analysis. It performs Step 2 of the SAIGE-GENE+ association tests for each variant set, where each set has exactly one variant removed from the full collection. The script iterates over all annotation files, with each file representing a reduced variant set missing one specific variant. By looping through these files, the pipeline re-runs Step 2 for every leave-one-out subset, allowing for evaluation of each variant’s individual contribution to the overall gene-based association signal and identifying variants that disproportionately influence the test statistic.

In [9]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261025 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [10]:
import os
import tempfile
import subprocess

# Create a separate Cloud SDK configuration directory
os.environ["CLOUDSDK_CONFIG"] = tempfile.mkdtemp(
        prefix="gcloud-dsub-",
        dir="/tmp",
)

print("CLOUDSDK_CONFIG:", os.environ["CLOUDSDK_CONFIG"])
print("CLOUDSDK_CONFIG =", os.environ.get("CLOUDSDK_CONFIG", "(default)"))
print("HOME =", os.environ.get("HOME"))

subprocess.run(
        ["gcloud", "config", "get-value", "account"],
        check=False,
)

CLOUDSDK_CONFIG: /tmp/gcloud-dsub-aaoia_n0
CLOUDSDK_CONFIG = /tmp/gcloud-dsub-aaoia_n0
HOME = /home/dataproc
pet-27662581260322f686e2a@wb-fast-radish-6415.iam.gserviceaccount.com


CompletedProcess(args=['gcloud', 'config', 'get-value', 'account'], returncode=0)

# COMMAND 

In [11]:
%%writefile filtered_high_confidence_analysis.sh

#!/bin/bash
set -o errexit
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_biosample_age; do 

            echo "Processing population: ${pop} | phenotype: ${pheno} | chr: ${chrom}"
        
            step2_SPAtests.R \
                --bedFile="${plink_bed}" \
                --bimFile="${plink_bim}" \
                --famFile="${plink_fam}" \
                --SAIGEOutputFile="${output}/${pop}_${pheno}_ex_hotspots_AB_0.35_DP_20_high_confidence_chr${chrom}" \
                --LOCO=FALSE \
                --AlleleOrder=alt-first \
                --minMAF=0 \
                --minMAC=0.5 \
                --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
                --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --groupFile="${vat}/HC-annotation-chr${chrom}.txt" \
                --idstoIncludeFile="${keep}" \
                --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
                --maxMAF_in_groupTest=0.0001,0.001,0.01 \
                --is_output_markerList_in_groupTest=TRUE \
                --is_fastTest=TRUE
    done
done


Overwriting filtered_high_confidence_analysis.sh


# SUBMIT COMMAND 

In [12]:
%%bash --out saige_gene_step0

for chromo in 2 4 15; do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 4 \
  --machine-type n1-standard-1 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_updated_ex_hotspots_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_updated_ex_hotspots_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_updated_ex_hotspots_chr${chromo}.fam" \
  --input keep="${WORKSPACE_BUCKET}/data/chip/hotspots/keep.txt" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_HC_annotation/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/chip_sensitivity/" \
  --env chrom=${chromo} \
  --script "filtered_high_confidence_analysis.sh"

done

Job properties:
  job-id: filtered-h--vienose--261026-132017-73
  job-name: filtered-high-confidence-analysis
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/filtered-h--vienose--261026-132017-73-0-0
Launched job-id: filtered-h--vienose--261026-132017-73
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'filtered-h--vienose--261026-132017-73' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'filtered-h--vienose--261026-132017-73' --users 'vienose'
Job properties:
  job-id: filtered-h--vienose--261026-132019-39
  job-name: filtered-high-confidence-analysis
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/filtered-h--vienose--261026-132019-39-0-0
Launched job-id: filtered-h--vienose--261026-132019-39
To che

In [6]:
!dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'filtered-h--vienose--261026-131405-45' --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
filtered-hig...  SUCCESS   10-26 13:18:11



In [16]:
!dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'filtered-h--vienose--261026-132021-11' --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
filtered-hig...  SUCCESS   10-26 13:24:36

